# RAG Smoke Test Notebook

This notebook assumes you have already built persistent **Chroma** indexes (e.g., via `build_indexes.py`) under:

- `data/indexes/TPMS`
- `data/indexes/AA`

It will:
1) Load your local retriever (`rag.py`)  
2) Run **retrieval-only** tests (inspect top-k chunks)  
3) Run **full RAG** tests (retrieve → answer with citations)


In [1]:
# If needed, uncomment and run:
# %pip install -q langchain langchain-community langchain-openai chromadb python-dotenv

import os
import sys
import time
from pathlib import Path

print("Python:", sys.version)
print("Platform:", sys.platform)
print("CWD:", os.getcwd())


Python: 3.13.5 (main, Jun 11 2025, 15:36:57) [Clang 17.0.0 (clang-1700.0.13.3)]
Platform: darwin
CWD: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/scripts


In [2]:
# Load environment variables (expects OPENAI_API_KEY)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded .env (if present).")
except Exception as e:
    print("dotenv not available or failed to load:", e)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing. Set it in your environment or in a .env file."
print("OPENAI_API_KEY is set ✅")


Loaded .env (if present).
OPENAI_API_KEY is set ✅


In [15]:
from pathlib import Path
import sys

# Where am I running from?
HERE = Path.cwd().resolve()
print("Running from:", HERE)

# Find repo root by walking up until we see "data/indexes" (or "pyproject.toml")
PROJECT_ROOT = next(
    p for p in [HERE, *HERE.parents]
    if (p / "data" / "indexes").exists() or (p / "pyproject.toml").exists() or (p / "app").exists()
)

# ✅ correct base regardless of where notebook is run from
INDEX_BASE = PROJECT_ROOT / "data" / "indexes"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INDEX_BASE:", INDEX_BASE)
print("INDEX_BASE exists?", INDEX_BASE.exists())
    
sys.path.insert(0, str(PROJECT_ROOT))

Running from: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/scripts
PROJECT_ROOT: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot
INDEX_BASE: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/data/indexes
INDEX_BASE exists? True


In [ ]:

from app.nodes import rag

print("Imported rag.py from:", rag.__file__)
print("Project root:", PROJECT_ROOT)
print("Embedding model:", getattr(rag, "EMBED_MODEL", None))

Imported rag.py from: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/app/nodes/rag.py
Project root: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot
Embedding model: text-embedding-3-small


In [18]:
PRODUCTS = ["TPMS", "AA"]
DEFAULT_K = 5

missing = [p for p in PRODUCTS if not (INDEX_BASE / p).exists()]
if missing:
    print("⚠️ Missing index folders for:", missing)
    print("Expected paths like:", [str(INDEX_BASE / p) for p in missing])
else:
    print("All index folders found ✅")


# Simple queries to use for smoke tests (edit freely)
SMOKE_QUERIES = {
    "TPMS": [
        "Cómo se rotan los sensores y cuándo conviene hacerlo?",
        "Qué hago si un sensor no aparece en la pantalla?",
        "Es 433mhz? Cómo verifico frecuencia y compatibilidad?"
    ],
    "AA": [
        "Qué temperatura de salida tiene el HDK 2200?",
        "Cuánto consumo tiene un aire 2200 24V aproximadamente?",
        "Cuántas horas puedo usar el aire con el motor apagado?"
    ],
}

All index folders found ✅


In [19]:
# Helper: robust retrieval call across LangChain versions
from typing import List

def retrieve_docs(retriever, query: str) -> List:
    # Newer LC: retriever.invoke(query)
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    # Older LC: retriever.get_relevant_documents(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise TypeError("Unsupported retriever interface")

def pretty_doc(doc, max_chars: int = 420) -> str:
    meta = getattr(doc, "metadata", {}) or {}
    src = meta.get("source") or meta.get("file_path") or meta.get("path") or ""
    page = meta.get("page", meta.get("page_number", ""))
    head = f"source={src} page={page}".strip()
    txt = (getattr(doc, "page_content", "") or "").strip().replace("\n", " ")
    if len(txt) > max_chars:
        txt = txt[:max_chars] + "…"
    return f"{head}\n{txt}"


## 1) Retrieval-only smoke test

This prints the **top-k chunks** returned by the retriever so you can validate:
- The index loads correctly
- The docs look relevant
- Metadata looks reasonable (source paths, etc.)


In [20]:
for product_id in PRODUCTS:
    print("\n" + "="*90)
    print("RETRIEVAL TEST:", product_id)
    print("="*90)

    retriever = rag.get_retriever(product_id=product_id, k=DEFAULT_K)

    queries = SMOKE_QUERIES.get(product_id, ["Explain the main troubleshooting steps."])
    for q in queries:
        t0 = time.perf_counter()
        docs = retrieve_docs(retriever, q)
        dt = (time.perf_counter() - t0) * 1000

        print(f"\nQ: {q}")
        print(f"Retrieved {len(docs)} docs in {dt:.1f} ms")
        for i, d in enumerate(docs, 1):
            print(f"\n--- doc {i} ---")
            print(pretty_doc(d))



RETRIEVAL TEST: TPMS


/Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/app/nodes/rag.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs = Chroma(



Q: Cómo se rotan los sensores y cuándo conviene hacerlo?
Retrieved 0 docs in 1052.2 ms

Q: Qué hago si un sensor no aparece en la pantalla?
Retrieved 0 docs in 739.4 ms

Q: Es 433mhz? Cómo verifico frecuencia y compatibilidad?
Retrieved 0 docs in 354.5 ms

RETRIEVAL TEST: AA

Q: Qué temperatura de salida tiene el HDK 2200?
Retrieved 0 docs in 340.1 ms

Q: Cuánto consumo tiene un aire 2200 24V aproximadamente?
Retrieved 0 docs in 337.0 ms

Q: Cuántas horas puedo usar el aire con el motor apagado?
Retrieved 0 docs in 298.7 ms


## 2) Full RAG test (retrieve → answer)

This uses:
- `rag.get_retriever()` for retrieval
- `ChatOpenAI` for generation
- A simple prompt that forces citation markers like `[1] [2]` pointing to retrieved chunks


In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def format_context(docs, max_chars_each: int = 1200) -> str:
    blocks = []
    for i, d in enumerate(docs, 1):
        meta = getattr(d, "metadata", {}) or {}
        src = meta.get("source") or meta.get("file_path") or meta.get("path") or f"chunk_{i}"
        txt = (getattr(d, "page_content", "") or "").strip()
        if len(txt) > max_chars_each:
            txt = txt[:max_chars_each] + "…"
        blocks.append(f"[{i}] SOURCE: {src}\n{txt}")
    return "\n\n".join(blocks)

SYSTEM_PROMPT = '''You are a technical support assistant.
Answer the user's question using ONLY the provided context.
If the context is insufficient, say so and ask 1-2 precise follow-up questions.
Always include citations like [1], [2] pointing to the context chunks you used.
Answer in Spanish (Rioplatense, clear and practical).'''

def rag_answer(product_id: str, question: str, k: int = 5, model: str = "gpt-4o-mini", temperature: float = 0.0):
    retriever = rag.get_retriever(product_id=product_id, k=k)

    t0 = time.perf_counter()
    docs = retrieve_docs(retriever, question)
    t_retr = (time.perf_counter() - t0) * 1000

    context = format_context(docs)

    llm = ChatOpenAI(model=model, temperature=temperature)

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"PRODUCT: {product_id}\n\nCONTEXT:\n{context}\n\nQUESTION:\n{question}")
    ]

    t1 = time.perf_counter()
    resp = llm.invoke(messages)
    t_gen = (time.perf_counter() - t1) * 1000

    return {
        "answer": getattr(resp, "content", str(resp)),
        "docs": docs,
        "timing_ms": {"retrieve": t_retr, "generate": t_gen, "total": t_retr + t_gen}
    }


In [22]:
# Run a couple of RAG examples per product
for product_id in PRODUCTS:
    print("\n" + "="*90)
    print("RAG TEST:", product_id)
    print("="*90)

    for q in SMOKE_QUERIES.get(product_id, [])[:2]:
        out = rag_answer(product_id, q, k=DEFAULT_K)
        print(f"\nQ: {q}")
        print("Timing (ms):", out["timing_ms"])
        print("\nA:\n", out["answer"])

        # Show the sources returned
        print("\nSources returned (top-k):")
        for i, d in enumerate(out["docs"], 1):
            meta = getattr(d, "metadata", {}) or {}
            print(f"  [{i}] {meta.get('source') or meta.get('file_path') or meta.get('path') or 'unknown'}")



RAG TEST: TPMS

Q: Cómo se rotan los sensores y cuándo conviene hacerlo?
Timing (ms): {'retrieve': 372.18333399505354, 'generate': 2353.9606659905985, 'total': 2726.143999985652}

A:
 El contexto proporcionado no incluye información específica sobre la rotación de los sensores TPMS (Sistema de Monitoreo de Presión de Neumáticos). Para poder ayudarte mejor, ¿podrías especificar si te refieres a la rotación de los sensores en relación a los neumáticos o a otro aspecto? Además, ¿estás buscando información sobre cuándo es recomendable hacer esta rotación?

Sources returned (top-k):

Q: Qué hago si un sensor no aparece en la pantalla?
Timing (ms): {'retrieve': 367.21695799496956, 'generate': 2572.7039169869386, 'total': 2939.920874981908}

A:
 Si un sensor no aparece en la pantalla del TPMS, puede ser útil seguir estos pasos:

1. **Verificar la conexión**: Asegúrate de que el sensor esté correctamente instalado y conectado.
2. **Reiniciar el sistema**: A veces, reiniciar el sistema del TPM

## 3) Interactive quick test

Edit `product_id` and `question` and re-run the cell to test new queries quickly.


In [ ]:
product_id = "TPMS"  # "AA"
question = "El primer día andaban los 4 sensores, al día siguiente faltaban 2 y luego aparecen. A qué se puede deber y qué pruebo?"

out = rag_answer(product_id, question, k=5)
print("Timing (ms):", out["timing_ms"])
print("\nANSWER:\n", out["answer"])

print("\n--- Retrieved context (for debugging) ---\n")
for i, d in enumerate(out["docs"], 1):
    print(f"\n===== CHUNK [{i}] =====")
    print(pretty_doc(d, max_chars=900))


## 4) Common troubleshooting checklist (if things fail)

- `OPENAI_API_KEY` set and valid  
- `data/indexes/<PRODUCT>` exists and contains Chroma persisted files  
- `collection_name` matches: `f"{product_id}_docs"` (your code uses this convention)  
- Your knowledge docs are in `knowledge/<PRODUCT>/*.docx` (only needed when rebuilding indexes)

If you changed any naming conventions, update them in `rag.py`.
